# Backtest Engine — Full Validation Suite

Parameter heatmaps, time splits, Monte Carlo, stress tests, walk-forward — all strategies.

In [ ]:
%matplotlib inline
import sys, os
from pathlib import Path
_root = Path(os.getcwd())
while not ((_root / 'src').exists() and (_root / 'pyproject.toml').exists()) and _root != _root.parent:
    _root = _root.parent
sys.path.insert(0, str(_root))


## 1. Generate Data + Config

In [ ]:
np.random.seed(42)
n = 2000
dates = pd.date_range('2019-01-01', periods=n, freq='D')
mu, sigma = 0.0003, 0.012
returns = np.random.randn(n) * sigma + mu
returns[800:950] -= 0.004
returns[1500:1700] += 0.005
price = 100 * np.exp(np.cumsum(returns))

df = pd.DataFrame({
    'timestamp': dates, 'close': price,
    'open': price * (1 + np.random.randn(n) * 0.002),
    'high': price * (1 + np.abs(np.random.randn(n) * 0.003)),
    'low': price * (1 - np.abs(np.random.randn(n) * 0.003)),
})
split = int(n * 0.7)
df_train, df_test = df.iloc[:split], df.iloc[split:]

config = BacktestConfig(initial_cash=10000, fee_bps=4, slippage_bps=1, periods_per_year=252)

pd.DataFrame({'':['train_days','test_days','start','end'],'value':[len(df_train),len(df_test),str(dates[0].date()),str(dates[-1].date())]})

## 2. Parameter Heatmaps — All Strategies

In [ ]:
strategy_grids = {
    'SMA Cross': (SmaCrossStrategy, {'fast_window': [5, 10, 20, 50, 100], 'slow_window': [50, 100, 200]}),
    'Mean Reversion': (MeanReversionStrategy, {'lookback': [20, 50, 100], 'z_entry': [1.0, 1.5, 2.0], 'z_exit': [0.3, 0.5]}),
    'Regime Router': (RegimeRouterStrategy, {'vol_window': [10, 20, 30, 50], 'mom_window': [3, 5, 10, 20], 'regime_percentile': [40, 50, 60]}),
    'S1 Hard70': (S1Hard70Strategy, {'vol_window': [10, 20, 30], 'mom_window': [3, 5, 10], 'gate_percentile': [60, 70, 80]}),
}

heatmap_results = {}
fig, axes = plt.subplots(2, 2, figsize=(24, 18))
fig.patch.set_facecolor(BG)

for ax, (name, (cls, grid)) in zip(axes.flat, strategy_grids.items()):
    gs = grid_search(df_train, cls, grid, config=config, metric='sharpe', verbose=False)
    heatmap_results[name] = gs
    
    keys = list(gs.best_params.keys())
    pivot = gs.all_results.pivot_table(values='_score', index=keys[0], columns=keys[1] if len(keys)>1 else keys[0], aggfunc='mean')
    
    im = ax.imshow(pivot.values, aspect='auto', cmap='magma')
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            ax.text(j, i, f'{pivot.values[i,j]:.2f}', ha='center', va='center', color='white', fontsize=8, fontweight='bold')
    ax.set_xticks(range(len(pivot.columns))); ax.set_xticklabels(pivot.columns, color='white', fontsize=8)
    ax.set_yticks(range(len(pivot.index))); ax.set_yticklabels(pivot.index, color='white', fontsize=8)
    ax.set_title(f'{name}\nBest: {gs.best_params} | Sharpe={gs.best_score:.3f}', color='white', fontsize=10, fontweight='bold')
    ax.set_xlabel(keys[1] if len(keys)>1 else keys[0], color='white', fontsize=8)
    ax.set_ylabel(keys[0], color='white', fontsize=8)
    
fig.tight_layout(pad=3)
fig.savefig(_root/'images'/'validation_heatmaps.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 3. Time Split Validation — 5 Folds per Strategy

In [ ]:
def time_split_test(df, strategy_class, param_grid, n_splits=5):
    n = len(df); fold_size = n // (n_splits + 1)
    results = []
    for i in range(n_splits):
        train_end = (i+1)*fold_size; test_start = train_end
        test_end = min(test_start+fold_size, n)
        train_df = df.iloc[:train_end]; test_df = df.iloc[test_start:test_end]
        if len(train_df)<100 or len(test_df)<20: continue
        gs = grid_search(train_df, strategy_class, param_grid, config=config, verbose=False)
        best = strategy_class(**gs.best_params)
        r = BacktestEngine(config).run(test_df, best)
        m = r.metrics
        results.append({'fold':i+1,'train_days':len(train_df),'test_days':len(test_df),
                        'best_params':gs.best_params,'train_sharpe':gs.best_score,
                        'test_sharpe':m['sharpe'],'test_return_pct':m['total_return_pct'],
                        'bh_return_pct':m['bh_return_pct'],'excess_pct':m['excess_return_pct'],
                        'alpha_pct':m['alpha_pct'],'beta':m['beta'],'max_dd_pct':m['max_drawdown_pct']})
    return pd.DataFrame(results)

all_ts = []
for name, (cls, grid) in strategy_grids.items():
    ts = time_split_test(df, cls, grid)
    ts['model'] = name
    all_ts.append(ts)

ts_all = pd.concat(all_ts, ignore_index=True)

fig, axes = plt.subplots(2, 2, figsize=(24, 14))
fig.patch.set_facecolor(BG)
for ax, name in zip(axes.flat, strategy_grids.keys()):
    sub = ts_all[ts_all['model'] == name]
    if sub.empty: continue
    x = np.arange(len(sub)); w = 0.25
    ax.bar(x-w, sub['test_sharpe'].values, w, color=GOLD, alpha=0.85, label='Sharpe')
    ax.bar(x, sub['excess_pct'].values, w, color=CYAN, alpha=0.85, label='Excess %')
    dd_vals = [-abs(v) for v in sub['max_dd_pct'].values]
    ax.bar(x+w, dd_vals, w, color=RED, alpha=0.5, label='Max DD %')
    ax.set_xticks(x); ax.set_xticklabels([f'F{i+1}' for i in range(len(sub))], color='white', fontsize=8)
    ax.axhline(0, color='white', lw=0.5)
    ax.set_title(f'{name} — avg Sharpe={sub["test_sharpe"].mean():.3f} avg Excess={sub["excess_pct"].mean():+.1f}%', color='white', fontsize=10, fontweight='bold')
    ax.set_facecolor(PANEL); ax.tick_params(colors='white', labelsize=8)
    ax.grid(True, alpha=0.12, axis='y'); ax.legend(fontsize=7, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper right')

fig.suptitle('Time Split Validation — 5 Expanding Windows', color='white', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(_root/'images'/'validation_timesplit.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

ts_summary = ts_all.groupby('model').agg({'test_sharpe':'mean','excess_pct':'mean','alpha_pct':'mean','max_dd_pct':'mean'}).round(3)
ts_summary.columns = ['avg_sharpe','avg_excess_pct','avg_alpha_pct','avg_max_dd_pct']
ts_summary

## 4. Monte Carlo — Best Strategy

In [ ]:
best_name = ts_summary['avg_excess_pct'].idxmax()
best_cls, best_grid = strategy_grids[best_name]
gs_best = heatmap_results[best_name]
best_strat = best_cls(**gs_best.best_params)

mc = monte_carlo_simulate(df_test, best_strat, MonteCarloConfig(n_simulations=200, noise_bps=8, delay_bars=(0,3), seed=42), config, verbose=False)

base = mc.base_equity.set_index('timestamp')['equity']
pcts = mc.percentile_curves

fig, ax = plt.subplots(figsize=(20, 10))
fig.patch.set_facecolor(BG)
ax.fill_between(base.index, pcts['p5'], pcts['p95'], color=GOLD, alpha=0.06, label='p5-p95')
ax.fill_between(base.index, pcts['p25'], pcts['p75'], color=GOLD, alpha=0.10, label='p25-p75')
ax.plot(base.index, pcts['p50'], color=GOLD, lw=1, alpha=0.4, ls='--', label='Median')
ax.plot(base.index, base.values, color=GOLD, lw=2.5, label='Base')
ax.axhline(config.initial_cash, color='white', ls='--', alpha=0.15)
ax.set_title(f'Monte Carlo — 200 paths | {best_name} (best={gs_best.best_params})', color='white', fontsize=13, fontweight='bold')
ax.set_facecolor(PANEL); ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12); ax.legend(fontsize=10, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')
fig.tight_layout()
fig.savefig(_root/'images'/'validation_montecarlo.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

mc.summary_metrics.head(16)

## 5. Stress Tests — All Strategies

In [ ]:
stress_results = {}
for name, (cls, grid) in strategy_grids.items():
    gs = heatmap_results[name]
    best_s = cls(**gs.best_params)
    st = run_stress_tests(df, best_s, config=config, verbose=False)
    stress_results[name] = st

fig, axes = plt.subplots(2, 2, figsize=(24, 14))
fig.patch.set_facecolor(BG)

for ax, name in zip(axes.flat, strategy_grids.keys()):
    sdf = stress_results[name].summary.dropna(subset=['return_pct'])
    if sdf.empty: continue
    y_pos = np.arange(len(sdf)); rets = sdf['return_pct'].values
    colors = [GREEN if v>=0 else RED for v in rets]
    ax.barh(y_pos, rets, color=colors, alpha=0.85, edgecolor='white', lw=0.3)
    ax.set_yticks(y_pos); ax.set_yticklabels(sdf['scenario'], color='white', fontsize=8)
    ax.axvline(0, color='white', lw=0.5)
    for i, v in enumerate(rets):
        ax.text(v+(0.3 if v>=0 else -2.5), i, f'{v:.1f}%', va='center', color='white', fontsize=7)
    ax.set_title(f'{name}', color='white', fontsize=10, fontweight='bold')
    ax.set_facecolor(PANEL); ax.tick_params(colors='white', labelsize=8)
    ax.grid(True, alpha=0.12, axis='x')

fig.suptitle('Stress Tests — All Strategies', color='white', fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig(_root/'images'/'validation_stress.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

## 6. Walk-Forward — Best Strategy

In [ ]:
wf = walk_forward_optimize(
    df, best_cls, best_grid,
    wf_config=WalkForwardConfig(train_window=300, test_window=100, min_train=100, purge_bars=10),
    metric='sharpe', engine_config=config, verbose=False
)

folds = wf.fold_results
fig, ax = plt.subplots(figsize=(20, 7))
fig.patch.set_facecolor(BG)
x = np.arange(len(folds)); w = 0.25
s_vals = [f.get('test_sharpe',0) for f in folds]
r_vals = [f.get('test_total_return_pct',0) for f in folds]
d_vals = [-abs(f.get('test_max_drawdown_pct',0)) for f in folds]
ax.bar(x-w, s_vals, w, color=GOLD, alpha=0.85, label='Sharpe')
ax.bar(x, r_vals, w, color=CYAN, alpha=0.85, label='Return %')
ax.bar(x+w, d_vals, w, color=RED, alpha=0.5, label='Max DD %')
ax.set_xticks(x); ax.set_xticklabels([f'F{f["fold"]}' for f in folds], color='white', fontsize=7, rotation=45)
ax.axhline(0, color='white', lw=0.5)
ax.set_title(f'Walk-Forward — {best_name} ({len(folds)} folds) | avg Sharpe={wf.aggregate_metrics.get("avg_sharpe",0):.3f}', color='white', fontsize=13, fontweight='bold')
ax.set_facecolor(PANEL); ax.tick_params(colors='white', labelsize=8)
ax.grid(True, alpha=0.12, axis='y'); ax.legend(fontsize=9, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper right')
fig.tight_layout()
fig.savefig(_root/'images'/'validation_walkforward.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

pd.DataFrame(folds)[['fold','best_params','test_sharpe','test_total_return_pct','test_max_drawdown_pct']].head(10)

## 7. Full Metrics Comparison — Best Strategy on Test Set

In [ ]:
r = BacktestEngine(config).run(df_test, best_strat)

bh = df_test.set_index('timestamp')['close'] / df_test['close'].iloc[0] * config.initial_cash
eq = r.equity_curve.set_index('timestamp')['equity']
dd = (eq / eq.cummax() - 1) * 100

fig, ax = plt.subplots(figsize=(20, 10))
fig.patch.set_facecolor(BG)
ax.plot(bh.index, bh.values, color=WHITE, lw=1.2, ls='--', alpha=0.5, label=f'Buy & Hold ({r.metrics["bh_return_pct"]:+.1f}%)')
ax.plot(eq.index, eq.values, color=GOLD, lw=2.5, label=f'{best_name} ({r.metrics["total_return_pct"]:+.1f}%)')
ax.fill_between(eq.index, eq.values, config.initial_cash, where=eq.values>=config.initial_cash, color=GREEN, alpha=0.06)
ax.fill_between(eq.index, eq.values, config.initial_cash, where=eq.values<config.initial_cash, color=RED, alpha=0.06)
ax.axhline(config.initial_cash, color='white', ls='--', alpha=0.15)
m = r.metrics
ax.set_title(f'{best_name} vs BH — α={m["alpha_pct"]:+.1f}% β={m["beta"]:.2f} Sharpe={m["sharpe"]:.2f} MaxDD={m["max_drawdown_pct"]:.1f}%', color='white', fontsize=12, fontweight='bold')
ax.set_facecolor(PANEL); ax.tick_params(colors='white', labelsize=9)
ax.grid(True, alpha=0.12); ax.legend(fontsize=11, facecolor=PANEL, edgecolor='white', labelcolor='white', loc='upper left')
fig.tight_layout()
fig.savefig(_root/'images'/'validation_equity.png', dpi=300, facecolor=BG, edgecolor='none', bbox_inches='tight')
plt.show()

pd.Series({k: round(v, 2) if isinstance(v, float) else v for k, v in sorted(r.metrics.items())}).to_frame('value')

## Summary

| Validation | What it tests | Output |
|-----------|---------------|--------|
| Parameter Heatmaps | Grid search across 2-3 params, 4 strategies | `validation_heatmaps.png` |
| Time Split | 5 expanding windows, grid search per window | `validation_timesplit.png` |
| Monte Carlo | 200 perturbed paths (best strategy) | `validation_montecarlo.png` |
| Stress Tests | 8 crisis scenarios per strategy | `validation_stress.png` |
| Walk-Forward | Rolling train/test, purge+embargo | `validation_walkforward.png` |
| Equity Curve | Best strategy vs BH, full metrics | `validation_equity.png` |